# Magnetized Metalic Objects - Readme 

### How Domenico intends to extend these notes further

**Define first milestone in top-down approach** 
1. Arrive at 3D Galerkin MoM code with numerical evaluation on the interaction integrals. Add surface-surface and surface-volume interactions to existing 3D implemention; 

**Define second milestone in top-down approach**
1. implement dimension reduction for $P_{\alpha\beta}$ for $\alpha \approx \beta$ for a test case consisting of two tetrahedra only; 

**Define first milestone in bottom-up approach**
1. Arrive at 1D code with mass-matrix and convolution kernel (first regular, then singular) contribution. Add mass matrix. Replace sparse stiffness matrix by dense stiffness matrix. Can computed solution be verified symbolically (using e.g. sympy) or using a shooting method (using e.g. solve_bvp)? 

**Define second milestone in bottom-up approach**
1. Speculate on how to extend 1D code to 3D. 

**Criteria influencing the design of the data-structure holding the mesh**

1. we wish to iterate over cells in the volume and facets on the surface seperately. We thus require access to surface triangles.  
2. we wish to perform dimension reduction to evaluate the singular part of the interactio n integrals. We thus need to store facets and edges; 

We foresee two options to construct the mesh. 

1. using GMSH: using <i>gmsh.model.mesh.getElements(dim, tag)</i> with dim=3 for cells (tetrahedra) and with dim=2 for facets (triangles). Need to mark the surface elements in the mesh. Still requires functionality to extract facet-edge-node connectivity from the mesh. Relying on GMSH avoids introducing additional packages;      
2. using Ferrite.jl: using the cell iterator and the facet iterator that Ferrite provides. Ferrite.jl provides built-in mesh generator for uniform meshes. This allows to avoid the mesh IO that GMSH requires. Ferrite.jl also provides quadrature over the elements that we can take advantage of for elements at a large distance from each other. See notebook [mom_ferrite.ipynb](./mom_ferrite.ipynb); 

**The mesh data structure**

The mesh is a struct with the following five fields: 
1. a field <i>Elements</i>: holds an nnodes-array of type <i>Element</i>. The struct <i>Element</i> has a field nodes (4-array of integers), a field edges (6-array of integers), a field facets (4-array of integers), a field volume (Float64) and a 4-by-4  matrix of coefficients of the shape functions. Not sure whether using structs-as-functions to define the basis functions has a penalty; 
2. a field <i>Faces</i>: holds an nfaces-array of type <i>Face</i>. The struct <i>Face</i> has a field nodes (3-array of integers), a field edges (3-array of integers), a field surface (Float64), 3-by-3  matrix of coefficients of the shape functions (recomputed from scratch), tangential vectors, normal vector. Same question as before. Not sure how to extract relevant information from GMSH;  
3. a field <i>SurfaceFaces</i>: holds an nsurfaces-integer-array of global face indices lying on the surface. 
4. a field <i>Edges</i>: holds an nedges-array of type <i>Edge</i>. The struct <i>Edges</i> has a field nodes (2-array of integers), a field length (Float64) and a 2-by-2  matrix of coefficients of the shape functions (recompute from scratch). Same questions  as before. 
5. a field <i>Points</i>: holds an npoints-array of type <i>Point</i> (holds the memory). The struct <i>Point</i> is a static array of 3 Float64; 

The mesh data structure currently in place duplicates notes. This has to be removed. 

**Compute SurfaceFacet - SurfaceFacet interaction**

For 3 components of the magnetization, compute ($k$ here denotes the test function and therefore appears as column index) 

$$
S_{\alpha\beta}[\ell,k] = \int_{S_{\alpha}} \phi_M^{\partial \Omega} \, \phi_{k,n} \, dS_{\alpha} =  \frac{1}{4 \pi} \int_{S_{\alpha}} \int_{S_{\beta}} \frac{\phi_{\ell,n} \, \phi_{k,n}}{\|\mathbf{r}' - \mathbf{r} \|}  \, dS_{\beta} \, dS_{\alpha} \text{ for } 1 \leq \ell, k \leq 3  
$$

Zero contribution on facets where magnetization has a zero normal component. Numerator is polynomial of degree 2. Explicit for loop is required. 

1. for SurfaceFacet - SurfaceFacet we have 3 (number of Facet basis functions) * 3 (number of components of the magnetization vector) = 9 degrees of freedom. We thus need to compute a 9-by-9 matrix. 
2. the 4D integral can be computed numerically by calling the function <i>integrate_tri</i> twice; 
3. the computed contribution can be added to the already preallocated matrix; 

**Compute Tetrahedron - SurfaceFacet interaction**

$$
PS_{\alpha\beta}[\ell,k] =  - \int_{P_{\alpha}} \phi_M^{\partial \Omega} \left[ \text{div}_{\mathbf{r}} \,  \boldsymbol{\phi}_k \right] \, dP_{\alpha} \text{  for } 1 \leq \ell \leq 3 \text{ for } 1 \leq k \leq 4 
$$

**Excercise: Requires more information: Do later: Two Triangular Facets Test Case intended as a test case to compute the integral $I_{24}$ by reduction to lower dimension** 

More text here; 

**Extend double loop over elements to 4-fold loop over faces belonging to element or elementp (avoid nested constructions!)** 

More text here; 

**Post-processing** 
1. write output file as a VTKFile for unstructured tetrahedral grids using [WriteVTK.jl](https://juliavtk.github.io/WriteVTK.jl/dev/) . Use VTK_TETRA = 10. Include volumetric and surface contribution of the scalar potential in the post-processing; 
2. view output file using paraview; 

<b>To do</b>:
1. introduce [PhysicalConstants.jl](https://github.com/JuliaPhysics/PhysicalConstants.jl) 

## Section 1: Introduction 

When a metalic object is placed inside a magnetic field, a magnetization field is generated in the object. This magnetization opposes the external field. The object is said to be magnetized. The objective of this project is to compute the spatial distribution of the magnetization field inside the object. A coupled system of three partial differential equations for the components of the magnetization vector therefore needs to be solved numerically. These equations contains an integral term that renders this tasks challenging.     

The computation of the magnitization field has numerous practical applications. In the non-destructive testing of metallic object for instance, anomalies in the magnetic field distribution signal cracks or other defects. Other applications can be found in medical imaging, geophysical prospecting and magnetic field sensoring. 

Simulators for the magnization field have a long history of development. Traditional approaches are based on so-called collocation methods. These methods ressemble the finite difference method. The advantage of these methods is that the resulting linear system is straightforward to assemble. In the presence of the integral term in the partial differential equations, however, the resulting coefficient matrix becomes non-symmetric. This renders its large scale deployment cumbersome. 

The objective of this project is to contribute to the development of a novel simulation approach for the magnetization field inside metallic objects. Unlike traditional approaches, our appropach is based on a variational formulation. Our approach thus ressembles a finite element method. The system of integro-differential equations for the components of the magnetization field is discretized by a weighted residual method. The construction of the coefficient matrix of the resulting linear system requires the computation of the six-dimensional singular integrals that represent the source - receiver interactions. These integrals can be computed using an elegant analytical approach based on Euler's integration theorem for homogeneous functions. The linear systems that resuls from our approach are symmetric and positive definite. It is therefore easy to solve. 

The approach we suggest here is expected to render the computation of the magnetization field in realistic applications feasible.    

<b>Assembly of the linear system</b>: The focus of the project is placed on the assembly of the linear system. A computationally efficient procedure to treat the six-dimensional volume(3D)-volume(3D) interaction integrals will have to be developed. We will use the Euler Integration Theorem (a generalization of the Green-Gauss divergence theorem) for homogeneous functions.  

<b>Solve of the linear system</b>: Once the linear system is solved, it can be solved by a direct linear solveer for symmetric and positive definite linear system. Such solver are based on a Choleskly decomposition of the coefficient matrix.  

(Insert figures here).

## Section 2: Notation, Concepts and Mathematical Preliminaries 

**Notation** Assume $P_{\alpha}$ and $P_{\beta}$ to be two disjoint tetrahedra, i.e., two elements in a conformal mesh of a 3D volume $\Omega$. The we will denote the six-dimensional (3D)volume-(3D)volume interaction integral $P_{\alpha\beta}$ as 

$$
P_{\alpha\beta} = \int_{P_{\alpha}} \int_{P_{\beta}} \frac{1}{\|\mathbf{r}' - \mathbf{r} \|} dP_{\beta} \, dP_{\alpha} \, . 
$$

Note that the integral is singular in case that $\alpha = \beta$. This singularity causes the numerical evaluation of this integral to be non-trivial. See [Singular_integral](https://en.wikipedia.org/wiki/Singular_integral) for concepts. Later, we will introduce similar interaction integrals between facets, edges and points.  

**Identities** We have that 

$$
\nabla' \cdot \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} = \frac{-2}{\| \mathbf{r} - \mathbf{r}' \|}
\text{ and }
\nabla \cdot \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} = \frac{2}{\| \mathbf{r} - \mathbf{r}' \|} \, . 
$$

**Application of Euler Integration Theorem for Homogeneous Functions** 

Assume $P_{\alpha}$ to be a tetrahedron bounded by 4 triangular facets $F_{\alpha} \in P_{\alpha}$. Then (add references here)

$$
\int_{P_{\alpha}} \frac{1}{\| {\mathbf r} - {\mathbf r}' \|} \, d\Omega = \frac{1}{2} \sum \int_{F_{\alpha} \in P_{\alpha}} \left[ {\mathbf n}({\mathbf r}) \cdot ({\mathbf r} - {\mathbf r}') \right] \frac{1}{\| {\mathbf r} - {\mathbf r}' \|} \, dS
$$

and 

$$
\int_{P_{\alpha}} \frac{{\mathbf r} - {\mathbf r}'}{\| {\mathbf r} - {\mathbf r}' \|} \, d\Omega = \frac{1}{3} \sum \int_{F_{\alpha} \in P_{\alpha}} \left[ {\mathbf n}({\mathbf r}) \cdot ({\mathbf r} - {\mathbf r}') \right] \frac{{\mathbf r} - {\mathbf r}'}{\| {\mathbf r} - {\mathbf r}' \|} \, dS 
$$

**Integration by parts** Assume $\Omega \subset \mathbb{R}^3$ to be a volume in ${\mathbb R}^3$ bounded by a surface $\partial \Omega$ with outward normal ${\mathbf n}$. Assume $f(\mathbf{r})$ to be a scalar field on $\Omega$. Assume $\mathbf{V}(\mathbf{r})$ to be a vector field on $\Omega$ with normal component $V_n(\mathbf{n}) = \mathbf{V}(\mathbf{r}) \cdot \mathbf{n}$ on $\partial \Omega$. Then 

$$
\text{div} \left[ f \, \mathbf{V}  \right] = 
\text{grad}(f) \cdot \mathbf{V} + f \, \text{div} \mathbf{V} \, . 
$$

On the other hand, by integration over $\Omega$ and applying the Green-Gauss divergence theorem, we obtain that 

$$
\int_{\Omega} \text{div} \left[ f \, \mathbf{V} \right] \, d\Omega = \int_{\partial \Omega} \left[ f \, \mathbf{V} \right] \cdot \mathbf{n} \, dS = \int_{\partial \Omega} f \, \left[ \mathbf{V} \cdot \mathbf{n} \right] \, dS = \int_{\partial \Omega} f \, V_n \, dS 
$$

By combining the two above results, we arrive at 

$$
\int_{\partial \Omega} f \, V_n \, dS = \int_{\Omega} \text{div} \left[ f \, \mathbf{V} \right] \, d\Omega = \int_{\Omega} \left[ \text{grad}(f) \cdot \mathbf{V} \right] \, d\Omega + \int_{\Omega} \left[ f \, \text{div} \mathbf{V} \right] \, d\Omega \, .  
$$

or 

$$
\int_{\Omega} \left[ \text{grad}(f) \cdot \mathbf{V} \right] \, d\Omega = \int_{\partial \Omega} f \, V_n \, dS - \int_{\Omega} \left[ f \, \text{div} \mathbf{V} \right] \, d\Omega \, . 
$$

## Section 3: Project Description 

### Section 1.3: Problem Formulation 

<b>Computational Domain</b> We assume that the computational domain $\Omega$ is a cube with lenght $L$, height $H$ and depth $D$ alligned with the coordinate axes. Then $0 \leq x \leq L$, $0 \leq y \leq H$ and $0 \leq z \leq D$ and $\Omega = [0,L] \times [0,H] \times [0,D]$. Typical values are $L = H = 1 \, \text{m}$ and $0.01 \, \text{m} \leq D \leq 0.1 \, \text{m}$. In thick plate, the magnetization will varry with the $z$-coordinates. In thin plates instead, this variation is expected to be small. 

We will distinguish between the interior and the boundary of $\Omega$. We therefore assume that $\Omega$ is open (the boundary of $\Omega$ does not belong to $\Omega$). We assume that $\partial \Omega$ (the union of the six (front, back, top, bottom, left and right) quadrilateral facets) denotes the boundary of $\Omega$. We assume that $\overline{\Omega} = \Omega \cup \partial \Omega$ (i.e. the union of the volume and its surface) denotes the closure of $\Omega$. We assume that the exterior to $\Omega$ is the volume ${\mathbb R}^3 \setminus \overline{\Omega}$. 

We assume that $\mathbf{r} = (x,y,z)$ denotes the position vector of a observer (destination) inside of $\overline{\Omega}$ (thus possibly on the boundary). We assume that $\mathbf{r}' = (x',y',z')$ denotes the position vector of a source inside of $\overline{\Omega}$. We will denote by $\nabla$ and $\nabla'$ the gradient operator wrt. $\mathbf{r}$ and $\mathbf{r}'$, respectively. We will use a similar notation for the divergence.   

<b>Alternative Computational Domain</b> Massive sphere. Analytical solution in spherical coordinates is available. 

<b>Material Properties</b> Assume that $\Omega$ has a [magnetic susceptibility](https://en.wikipedia.org/wiki/Magnetic_susceptibility) denoted by $\chi_{mag}$ and a relative [magnetic permeability](https://en.wikipedia.org/wiki/Permeability_(electromagnetism)) denoted by $\mu_r$. Then $\chi_{mag} = \mu_r -1$. When the volume $\Omega$ is placed inside an external magnetic field, it will magnetize. The volume $\Omega$ thus mimmics a metalic plate. Here it will be suffucient to assume that the plate is homogeneous and that therefore $\chi_{mag}$ is constant. Assume that $10 \leq \chi_{mag} \leq 1000$ (dimensionless, value for common steel types). Non-homogeneous plate can be modeled assuming that $\chi_{mag}$ is piecewise constant. Non-linear magnetization effects are excluded in this project.   

<b>Problem Description</b> To describe the problem, we introduce three [magnetic fields](https://en.wikipedia.org/wiki/Magnetic_field) (in units Ampere per meter). 

The external magnetic field will be denoted by ${\mathbf H}_{ext}$. This field causes the magnetization to be generated in $\Omega$. This field is assumed to be constant and equal in the inside and outside of $\Omega$. Then $\mathbf{H}_{ext}$ is a vector with three components . This can be expressed as $\mathbf{H}_{ext} = \left( H_{ext,x}, H_{ext,y}, H_{ext,z}\right)$ or as $ \mathbf{H}_{ext} = H_{ext,x} \mathbf{i} + H_{ext,y} \mathbf{j} + H_{ext,z} \mathbf{k}$. The orientation of $\Omega$ with respect to $\mathbf{H}_{ext}$ matters. We distinguish an in-plane and a perpendicular plane configuration. In an in-plane configuration, the plate $\Omega$ is alligned with the external field, e.g., with the $x$-axis. We can then write that $\mathbf{H}_{ext} = \left( H_{ext,x}, 0,0\right)$ with $H_{ext,x}$ a constant. In a perpendicular plane configuration, the plate $\Omega$ is oriented perpendicular to $\mathbf{H}_{ext}$. We can then write that $\mathbf{H}_{ext} = \left( 0, 0, H_{ext,z} \right)$ with $H_{ext,z}$ a constant. (Provide a representative value for the strength of the magnetic field. The earth magnetic field for instance is on average $50,000$ nanoTesla). 

The induced magnetic field will be denoted by ${\mathbf H}_{M}(\mathbf{r})$. This field is caused by the presence of a non-zero magnetization ${\mathbf M}(\mathbf{r})$ inside of $\Omega$. The relation between ${\mathbf M}(\mathbf{r})$ and ${\mathbf H}_{M}(\mathbf{r})$ can be expressed through the magnetic scalar potential $\phi_M(\mathbf{r})$ as 

$$
{\mathbf H}_{M}(\mathbf{r}) = - \text{grad}_{\mathbf{r}} \, \phi_M(\mathbf{r}) = - \nabla_{\mathbf{r}} \, \phi_M(\mathbf{r}) 
$$

(minus sign by convention). The scalar potential for ${\mathbf M}(\mathbf{r})$ inside of $\Omega$ (volumetric density $\rho_M = - \nabla_{\mathbf{r'}} \cdot \mathbf{M}(\mathbf{r}')$) and ${\mathbf M}(\mathbf{r})$ on $\partial \Omega$ (surface density $\rho_S = \mathbf{M}(\mathbf{r}') \cdot \mathbf{n}(\mathbf{r'})$) is given by 

$$
\begin{eqnarray}
\phi_M(\mathbf{r}) & = & \frac{1}{4 \pi}
\int_{\Omega} \frac{- \nabla_{\mathbf{r'}} \cdot \mathbf{M}(\mathbf{r}')}{ \|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' + \frac{1}{4 \pi} \int_{\partial \Omega} \frac{ \mathbf{M}(\mathbf{r}') \cdot \mathbf{n}(\mathbf{r'}) }{\|\mathbf{r}' - \mathbf{r} \|} \, dS' \nonumber \\
& = & \phi_M^{\Omega}(\mathbf{r}) + \phi_M^{\partial \Omega}(\mathbf{r}) \, . 
\end{eqnarray}
$$

(Note: unlike Eugene, we do not attach an index to $\mathbf{M}(\mathbf{r}')$. For us, the field to solve for is one and the same.) The total magnetic field will be denoted by ${\mathbf H}_{tot}(\mathbf{r})$. This field is the sum of the external and the induced magnetic field. We thus have that ${\mathbf H}_{ext} + {\mathbf H}_{M} = {\mathbf H}_{tot}$. The total field is equal to the external field in the exterior of $\overline{\Omega}$. The total field on $\overline{\Omega}$ is related to the magnetization via the constitutive equation ${\mathbf H}_{tot} = {\mathbf M}/\chi_{mag}$.

### Section 2.3: Vector Integro-Differential Equation for $\mathbf{M}(\mathbf{r})$ - Problem in Strong Form 

We wish to compute ${\mathbf M}(\mathbf{r})$ by solving a partial differential equation in which the external field ${\mathbf H}_{ext}$ acts as a source term (i.e., ${\mathbf H}_{ext}$ appears in the right-hand side of the equation). This differential equation can be written as  

$$
{\cal D} \left[ \mathbf{M}(\mathbf{r}) \right] = {\mathbf H}_{ext} \text{ on } \overline{\Omega}  
$$

where ${\cal D}$ is the 
[differential operator](https://en.wikipedia.org/wiki/Differential_operator) acting on ${\mathbf M}(\mathbf{r})$ as 

$$
\begin{eqnarray}
{\cal D}\left[ \mathbf{M}(\mathbf{r}) \right] & = & {\mathbf H}_{tot}(\mathbf{r}) - {\mathbf H}_{M}(\mathbf{r}) \nonumber \\ 
& = & \frac{1}{\chi_{mag}} \mathbf{M}(\mathbf{r}) +
\text{grad}_{\mathbf{r}} \left\{  
\frac{1}{4 \pi}  \int_{\Omega} \frac{- \text{div}_{\mathbf{r}} \mathbf{M}(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' + \frac{1}{4 \pi}  \int_{\partial \Omega} \frac{ \mathbf{M}(\mathbf{r}') \cdot \mathbf{n}(\mathbf{r'}) }{\|\mathbf{r}' - \mathbf{r} \|} \, dS' \right\} \nonumber \\
& = & \frac{1}{\chi_{mag}} \mathbf{M}(\mathbf{r}) + 
\nabla_{\mathbf{r}} \left\{ 
\frac{1}{4 \pi} \int_{\Omega} \frac{- \nabla_{\mathbf{r}} \cdot \mathbf{M}(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' + \frac{1}{4 \pi}  \int_{\partial \Omega} \frac{ \mathbf{M}(\mathbf{r}') \cdot \mathbf{n}(\mathbf{r'}) }{\|\mathbf{r}' - \mathbf{r} \|} \, dS' \right\} \, . \nonumber \\ 
& = & {\cal D}_1\left[ \mathbf{M}(\mathbf{r}) \right] + {\cal D}_2 \left[ \mathbf{M}(\mathbf{r}) \right] + {\cal B} \left[ \mathbf{M}(\mathbf{r}) \right] \nonumber 
\end{eqnarray}
$$

where ${\cal D}_1$ merely applies a scaling to $\mathbf{M}(\mathbf{r})$ (leading to a mass matrix for vector-valued problems as in classical FEM, see e.g. transient linear elasticity equations)

$$
{\cal D}_1\left[ \mathbf{M}(\mathbf{r}) \right] = \frac{1}{\chi_{mag}} \mathbf{M}(\mathbf{r}) \, , 
$$

where ${\cal D}_2$ is the  volume-integro-differential part of the equation (Note that after integration wrt. ${\mathbf r}'$ over $\Omega$, a function in ${\mathbf r}$ is obtained. This function can be differentiated wrt. ${\mathbf r}$.) (the ${\cal D}_2$ requires secoond order derivatives of $\mathbf{M}(\mathbf{r}$, thus requiring stringent smoothness requirements on $\mathbf{M}(\mathbf{r}$. Hence the term strong form.) (leading to a stiffness matrix. Unlike in classical FEM, this stiffness matrix will be dense due to integral term in the equations) 

$$
{\cal D}_2\left[ \mathbf{M}(\mathbf{r}) \right] = \text{grad}_{\mathbf{r}} \, \phi_M^{\Omega} = \frac{1}{4 \pi} \nabla_{\mathbf{r}} \, \int_{\Omega} \frac{ - \nabla' \cdot \mathbf{M}(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, , 
$$

and where ${\cal B}$ is the boundary-integro-differential part of the equation (Same observation as for ${\cal D}_2$, this time integrating over $\partial \Omega$.)

$$
{\cal B} \left[ \mathbf{M}(\mathbf{r}) \right] = \text{grad}_{\mathbf{r}} \, \phi_M^{\partial \Omega} = \, \frac{1}{4 \pi} \nabla_{\mathbf{r}} \int_{\partial \Omega} \frac{ \mathbf{M}(\mathbf{r}') \cdot \mathbf{n}(\mathbf{r'}) }{\|\mathbf{r}' - \mathbf{r} \|} \, dS' \, . 
$$

The differential equation that determines $\mathbf{M}(\mathbf{r})$ is an [integro-differential equation](https://en.wikipedia.org/wiki/Integro-differential_equation). The above integro-differential equation is a system of three coupled equations for the three components of the magnetization $\mathbf{M}(\mathbf{r})$. This equation is a vector-valued grad-div equation. See e.g [MFEM weak gradient example](https://mfem.org/fem_weak_form/) and 
[Moose grad-div example](https://mooseframework.inl.gov/syntax/MFEM/Grad-Div.html). (Not sure whether FENICS or dealII provides similar examples.) 

This equation for $\mathbf{M}(\mathbf{r})$ is solved on $\overline{\Omega}$ only. The magnetization $\mathbf{M}(\mathbf{r})$ remains zero outside of $\overline{\Omega}$. 

The kernel in ${\cal D}_2$ can be written as 

$$
{\cal K}(\mathbf{r},\mathbf{r}') = \frac{1}{4 \pi}  \frac{1}{\|\mathbf{r}' - \mathbf{r} \|} \, . 
$$

This kernel depends on the distance $\|\mathbf{r}' - \mathbf{r} \|$ only. Small values of this distance matter most. It is singular for $\mathbf{r}' = \mathbf{r}$. Add note on how to regularize the kernel. The convolution of $\mathbf{M}(\mathbf{r})$ and ${\cal K}(\mathbf{r},\mathbf{r}')$ can be written as 

$$
{\cal K}(\mathbf{r},\mathbf{r}') * \mathbf{M}(\mathbf{r}') = \frac{1}{4 \pi} \int_{\Omega} \frac{- \nabla_{\mathbf{r}'} \cdot \mathbf{M}(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, . 
$$

Link here to [Riesz_transform](https://en.wikipedia.org/wiki/Riesz_transform). 

We also write the integro-differential equation for $\mathbf{M}(\mathbf{r})$ in explicit form. The first of the three scalar equations is 

$$
\frac{1}{\chi_{mag}} M_x(\mathbf{r}) +  
\frac{\partial}{\partial x} \, \frac{1}{4 \pi}
\int_{\Omega} \frac{- \nabla_{\mathbf{r}'} \cdot \mathbf{M}(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' + 
\frac{\partial}{\partial x} \frac{1}{4 \pi}  \int_{\partial \Omega} \frac{ \mathbf{M}(\mathbf{r}') \cdot \mathbf{n}(\mathbf{r'}) }{\|\mathbf{r}' - \mathbf{r} \|} \, dS' = H_{ext,x} \, , 
$$

and simarly for $M_y(\mathbf{r})$ and $M_z(\mathbf{r})$.   

**Reference Solutions** Find reference solutions for in-plane and perpendicular-plane external fields using AI tools, literature, alternative softwarfe tools with or without regularizing the singularity of the interaction kernel. 

**Continuous Variational Formulation** Describe continuous variational formulation to arrive at estimates for a-posteriori error analysis that can be exploited in an h-adaptive or p-adaptive refinement strategy. Constant of continuity and coercivity.

### Section 3.3: Mesh Generation and Definition of Shape Functions 

We wish to compute the magnetization vector $\mathbf{M}(\mathbf{r})$ inside the metallic plate $\Omega$. We therefore generate a mesh on $\Omega$. 

<b>Mesh Generation</b> We denote the mesh by $\Omega^h$. Due to limitation currently faced in the analytical computation of the matrix elemdnts, we assume that the mesh consists of tetrahedral elements only. We denote the number of nodes, edges, facets and elements of $\Omega^h$ by $N_n$, $N_{ed}$, $N_f$ and $N_e$, respectively. Assume the nodes and the elements to be denoted by ${\mathbf x}_i$ for $1 \leq i \leq N_n$ and by $P_{\alpha}$ for $1 \leq \alpha \leq N_e$, respectively. The union of all elements $P_{\alpha}$ forms the entire domain of computation. That is, we have that $\cup P_{\alpha} | 1 \leq \alpha \leq N_e = \Omega$. We will use this elementary fact in the computation of the matrix $\underline{\underline{A}}$ and the right-hand side vector $\mathbf{b}$.    

We assume that the information to decompose an element $P_{\alpha}$ into a set of facets, a facet into a set of edges and an edge into a set of points to be available (representation of the mesh $\Omega^h$ as a directed a-cyclic graph (DAG) with corresponding operations to find parent nodes and child nodes. See e.g. [wikipedia entry on polygon mesh](https://en.wikipedia.org/wiki/Polygon_mesh).

<b>Shape Functions</b> The mesh $\Omega^h$ allows to define a set of nodal P1 linear Lagrange basis functions $\{ \phi_i(\mathbf{r}) | 1 \leq i \leq N _n \}$ (see classical finite elements method). The functions $\phi_i(\mathbf{r})$ are scalar functions. To be able to cast a vector-valued partial differential equation in weak or variational form, we define $3 \, N_n$ vector-valued shape functions $\boldsymbol{\phi}_{3\,i-2} = \left( \phi_i, 0,0\right)$ (weighting function for the PDE for $M_x({\mathbf r})$), 
$\boldsymbol{\phi}_{3i-1} = \left( 0, \phi_i, 0\right)$ (weighting function for $M_y({\mathbf r})$) and 
$\boldsymbol{\phi}_{3i} = \left( 0, 0, \phi_i \right)$ (weighting function for $M_z({\mathbf r})$ for $1 \leq i \leq N_n$.  

More information on the mesh generation is provided in the [notebook](./mom_3d_plate_gmsh.ipynb). 

**Definition of Shape Functions on a Single Tetrahedral Element** The tetrahedral mesh element $P_{\alpha}$ has $4$ nodes. Assume these nodes to be denote ${\mathbf x}_1$, ${\mathbf x}_2$, ${\mathbf x}_3$ and ${\mathbf x}_4$ (in local numbering on this single element). The linear Lagrange shape functions on this element can be expressed as $\phi_k({\mathbf r}) = a_k x + b_k y + c_k y + d_k$ for $1 \leq k \leq 4$, where $a_k$, $b_k$, $c_k$ and $d_k$ are coefficients. These coefficients are choosen such that the shape functions satisfy the constraint that $\phi_k({\mathbf x}_{\ell}) = \delta_{k\ell}$ for $1 \leq k,\ell \leq 4$. Linear system for these coefficients. 

**Divergence of Vector Shape Function** The divergence of the vector-valued shape functions can be expressed as 

$$
\nabla_{\mathbf{r}} \cdot \boldsymbol{\phi}_k = \frac{\partial}{\partial x} (a_k x + b_k y + c_k y + d_k) = a_k \text{ if } \mod(k,3) = 1 \\
\nabla_{\mathbf{r}} \cdot \boldsymbol{\phi}_k = \frac{\partial}{\partial y} (a_k x + b_k y + c_k y + d_k)= b_k \text{ if } \mod(k,3) = 2 \\
\nabla_{\mathbf{r}} \cdot \boldsymbol{\phi}_k = \frac{\partial}{\partial z} (a_k x + b_k y + c_k y + d_k)= c_k \text{ if } \mod(k,3) = 0
$$     

Observe that these computations are specific to linear shape functions on tetrahedral elements.    

### Section 4.3: Galerkin Method    
 
**Discrete Weak Form** The discrete weak form is obtained in two steps. First the integro-differential equation for $\mathbf{M}(\mathbf{r})$ in multiplied by a vector test function $\boldsymbol{\phi}_k$ with normal component $\boldsymbol{\phi}_k \cdot {\mathbf n} = \phi_{k,n}$. Next, integration over $\Omega$ is performed. (note: to be expanded with the continuous discrete weak form and to be expanded with the weak form for vector-valued equations using inner product with a vector-valued test function) 

$$
\int_{\Omega} {\cal D} \left[ \mathbf{M}(\mathbf{r}) \right] \cdot \boldsymbol{\phi}_k \, d\Omega = 
\int_{\Omega} \mathbf{H}_{ext} \cdot \boldsymbol{\phi}_k \, d\Omega 
\text{ for } \forall 1 \leq k \leq N_n 
$$ 

or after expanding ${\cal D} \left[ \mathbf{M}(\mathbf{r}) \right]$ in three terms we obtain $\forall 1 \leq k \leq N_n$

$$
\int_{\Omega} {\cal D}_1 \left[ \mathbf{M}(\mathbf{r}) \right] \cdot \boldsymbol{\phi}_k \, d\Omega + \int_{\Omega} {\cal D}_2 \left[ \mathbf{M}(\mathbf{r}) \right] \cdot \boldsymbol{\phi}_k \, d\Omega + \int_{\Omega} {\cal B} \left[ \mathbf{M}(\mathbf{r}) \right] \cdot \boldsymbol{\phi}_k \, d\Omega = 
\int_{\Omega} \mathbf{H}_{ext} \cdot \boldsymbol{\phi}_k \, d\Omega 
\hspace{1cm}  
$$ 

or after expanding ${\cal D}_1 \left[ \mathbf{M}(\mathbf{r}) \right]$, ${\cal D}_2 \left[ \mathbf{M}(\mathbf{r}) \right]$ and ${\cal B} \left[ \mathbf{M}(\mathbf{r}) \right]$ using their definitions we obtain $\forall 1 \leq k \leq N_n$

$$
\frac{1}{\chi_{mag}} \int_{\Omega} \mathbf{M}(\mathbf{r}) \cdot \boldsymbol{\phi}_k \, d\Omega + 
\int_{\Omega} \left[ \text{grad}_{\mathbf{r}} \, \phi_M^{\Omega} \right] \cdot \boldsymbol{\phi}_k \, d\Omega + \int_{\Omega} \left[ \text{grad}_{\mathbf{r}} \, \phi_M^{\partial \Omega}  \right] \cdot \boldsymbol{\phi}_k \, d\Omega = \int_{\Omega} \mathbf{H}_{ext} \cdot \boldsymbol{\phi}_k \, d\Omega
$$ 

and thus after applying integration by parts to the second and third term in the right-hand side of the equation   

$$
\frac{1}{\chi_{mag}} \int_{\Omega} \mathbf{M}(\mathbf{r}) \cdot \boldsymbol{\phi}_k \, d\Omega 
+ \int_{\partial \Omega} \phi_M^{\Omega} \, \phi_{k,n} \, dS
- \int_{\Omega} \phi_M^{\Omega} \left[ \text{div}_{\mathbf{r}} \, \boldsymbol{\phi}_k \right] \, d\Omega 
+ \int_{\partial \Omega} \phi_M^{\partial \Omega} \, \phi_{k,n} \, dS
- \int_{\Omega} \phi_M^{\partial \Omega} \left[ \text{div}_{\mathbf{r}} \,  \boldsymbol{\phi}_k \right] \, d\Omega 
= \int_{\Omega} \mathbf{H}_{ext} \cdot \boldsymbol{\phi}_k \, d\Omega \, . 
$$

This results in $3 \, N_n$ equations for the $3 \, N_n$ components of the vector of expansion coefficients $\mathbf{u}$. The mass matrix and load vector can be assembled by a loop over elements as in classical Galerlin FEM for vector-valued equations. The assembly of the stiffness matrix requires due care asc the integral term renders the stiffness matrix to be dense (as opposed to sparse in classical FEM).

**Expansion of Magnetization Components** The numerical approximation of the magnetization components can be expanded in as 

\begin{eqnarray}
M_x(\mathbf{r}) = \sum_{i=1}^{N_n} c_i \, \phi_i(\mathbf{r}) \\ 
M_y(\mathbf{r}) = \sum_{i=1}^{N_n} d_i \, \phi_i(\mathbf{r}) \\
M_z(\mathbf{r}) = \sum_{i=1}^{N_n} e_i \, \phi_i(\mathbf{r}) 
\end{eqnarray}

Note that components $M_x(\mathbf{r})$, $M_y(\mathbf{r})$ and $M_z(\mathbf{r})$ are expanded in the same basis with proper expansion coefficients. In case of linear Lagrange shape functions on tetrahedra, the element $P_{\alpha}$ has 4 local degrees of freediom per components, thus in total $3*4=12$ local degrees of freedom. Mapping from local to global degrees of freedom. The expansion coefficients $c_i$, $d_i$ and $e_i$ can stacked into a global vector $\mathbf{u}$ of dimension $3 \, N_n$ where $\mathbf{u}^T = \begin{pmatrix} \mathbf{c}^T \mathbf{d}^T \mathbf{e}^T \end{pmatrix}$. This global vector is the solution of the Galerkin-MoM linear system 

$$
\underline{\underline{A}} \, \mathbf{u} = \mathbf{b}
$$

where the matrix $\underline{\underline{A}} = \underline{\underline{A}}^{(1)} + \underline{\underline{A}}^{(2)}$ is a dense $3 \, N_n$-by-$3 \, N_n$ matrix consisting of 
mass matrix part $\underline{\underline{A}}^{(1)}$ and a stiffness part $\underline{\underline{A}}^{(2)}$. The mass matrix is the spatial discretization of the differential operator ${\cal D}_1 [\mathbf{M}(\mathbf{r})]$ and is very similar to a mass matrix in a classical finite element formulation. The stiffness matrix is the spatial discretization of the differential operator ${\cal D}_2 [\mathbf{M}(\mathbf{r})]$. Assembling this matrix is the main challenge of this project. 

**Elememt by Element Assembly** Here we describe the [Galerkin method](https://en.wikipedia.org/wiki/Galerkin_method) that allows to convert the system of integral partial differential-equations for $\mathbf{M}(\mathbf{r})$ into a weak or variational formulation. This variational formulation will allow a spatial discretization. See also example of the weak formulation of the Poisson equation as [wiki on weak formulation](https://en.wikipedia.org/wiki/Weak_formulation)). (Requires example of weak form of a c oupled system of differential equations.) 

Assume $g(\mathbf{r})$ and $h(\mathbf{r})$ to be scalar functions on $\Omega$ such that 

$$
g(\mathbf{r}) = \int_{\Omega} \frac{h(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, . 
$$

By decomposing $\Omega$ into a set of volumes $P_{\beta}$, we find that

$$
g(\mathbf{r}) = \int_{\Omega} \frac{h(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' = \sum_{\beta=1}^{N_e}  \int_{P_{\beta}} \frac{h(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, . 
$$

Then by decomposing $\Omega$ again into a set of volumes $P_{\alpha}$, the volume average of $g(\mathbf{r})$ can be written as  

$$
\begin{eqnarray}
\int_{\Omega} g(\mathbf{r}) \, d\Omega = 
\sum_{\alpha=1}^{N_e}  \int_{P_{\alpha}} g(\mathbf{r}) \, d\Omega 
& = & \sum_{\alpha,\beta=1}^{N_e}  \int_{P_{\alpha}} \int_{P_{\beta}} \frac{h(\mathbf{r}')}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, d\Omega \, . 
\end{eqnarray}
$$  

(Extend to weighted averaging of $g(\mathbf{r})$, Fourier analysius, expansion in sets of orthogonal functions.)

**Volume-volume interactions contributing to the stiffness matrix** Computing 6D integrals in a double loop over all tetrahedra. By writing $d\Omega = \sum_{\alpha} dP_{\alpha}$ and $d\Omega' = \sum_{\beta} dP_{\beta}$, we arrive at 

$$
\underline{\underline{A}}^{(2)} = \sum_{\alpha} \underline{\underline{A}}_{\alpha}^{(2)} = \sum_{\alpha\beta} \underline{\underline{A}}_{\alpha\beta}^{(2)} \text{ of size } 3 \, N_n \text{ by } 3 \, N_n 
$$

where the sum over $\alpha$ (destination) and $\beta$ (source) discretize the integral over $\mathbf{r}$ and $\mathbf{r}'$, respectively. In case that $\alpha = \beta$, self-interaction terms are considered. We have that 

$$
\underline{\underline{A}}_{\alpha\beta}^{(2)} = 
\begin{pmatrix} 
A_{\alpha\beta}^{xx} & A_{\alpha\beta}^{xy} &  A_{\alpha\beta}^{xz} \\
A_{\alpha\beta}^{yx} & A_{\alpha\beta}^{yy} &  A_{\alpha\beta}^{yz} \\ 
A_{\alpha\beta}^{zx} & A_{\alpha\beta}^{zy} &  A_{\alpha\beta}^{zz}
\end{pmatrix}
\text{ of size } 12 \text{ by } 12
$$

where (row determined by the expansion of the magnetization, columns determined by the test function). 

More explicitly we have for e.g. the $(1,1)$-block of $\underline{\underline{A}}_{\alpha\beta}^{(2)}$ that 

$$
A_{\alpha\beta}^{xx} = 
\begin{pmatrix} 
A_{\alpha\beta}^{xx,11} & A_{\alpha\beta}^{xx,12} & A_{\alpha\beta}^{xx,13} & A_{\alpha\beta}^{xx,14} \\
A_{\alpha\beta}^{xx,21} & A_{\alpha\beta}^{xx,22} & A_{\alpha\beta}^{xx,23} & A_{\alpha\beta}^{xx,24} \\
A_{\alpha\beta}^{xx,31} & A_{\alpha\beta}^{xx,32} & A_{\alpha\beta}^{xx,33} & A_{\alpha\beta}^{xx,34} \\
A_{\alpha\beta}^{xx,41} & A_{\alpha\beta}^{xx,42} & A_{\alpha\beta}^{xx,43} & A_{\alpha\beta}^{xx,44} 
\end{pmatrix}
\text{ of size } 4\text{ by } 4 
$$

where 

$$
\begin{eqnarray}
A_{\alpha\beta}^{xx,11} & =  & - \int_{P_{\alpha}} \frac{\partial \phi^{\alpha}_1({\mathbf r})}{\partial x} \int_{P_{\beta}} \frac{\partial \phi^{\beta}_1({\mathbf r}')}{\partial x'} \frac{1}{\|\mathbf{r}' - \mathbf{r} \|} \, dP_{\beta} \, dP_{\alpha}  \nonumber \\
& =  & - a^{\alpha}_1 \, a^{\beta}_1 \int_{P_{\alpha}} \int_{P_{\beta}} \frac{1}{\|\mathbf{r}' - \mathbf{r} \|} dP_{\beta} \, dP_{\alpha} \nonumber \\ 
& =  & - a^{\alpha}_1 \, a^{\beta}_1 \, P_{\alpha\beta} \nonumber
\end{eqnarray}
$$ 

Here we took advantage of the fact that the shape functions are linear to reduce the complexity of the computation. The factor $a_1^2$ take information on the shape functions into account. The factor $P_{\alpha\beta}$ takes information on the geometry of the tetrahedra $P_{\alpha}$ and $P_{\beta}$ into account. More generally, by varying the nodal index $i$ on $P_{\alpha}$ and $j$ on $P_{\beta}$, we have that 

$$
A_{\alpha\beta}^{xx,ij} = - a^{\alpha}_i \, a^{\alpha}_j \, P_{\alpha\beta} \text{ for } 1 \leq i,j \leq 4 \, . 
$$

The 4-by-4 matrix $A_{\alpha\beta}^{xx}$ can therefore be formed as the product of the scalar $P_{\alpha\beta}$ and the outer product of the two 4-by-1 vectors  

$$
{\mathbf a}^{\alpha} = \begin{pmatrix} a^{\alpha}_1 \\ a^{\alpha}_2 \\ a^{\alpha}_3 \\ a^{\alpha}_4 \end{pmatrix}
\text{ and }
{\mathbf a}^{\beta} = \begin{pmatrix} a^{\beta}_1 \\ a^{\beta}_2 \\ a^{\beta}_3 \\ a^{\beta}_4 \end{pmatrix} 
$$

as 

$$
A_{\alpha\beta}^{xx} = P_{\alpha\beta} \, {\mathbf a}^{\alpha} \, ({\mathbf a}^{\beta})^T \text{ of size } 4 \text{ by } 4 \, . 
$$

To generalize the above computations to the entire $\underline{\underline{A}}_{\alpha\beta}^{(2)}$ matrix (instead of merely the (1,1)-block), we define similarly to before 

$$
{\mathbf b}^{\alpha} = \begin{pmatrix} b^{\alpha}_1 \\ b^{\alpha}_2 \\ b^{\alpha}_3 \\ b^{\alpha}_4 \end{pmatrix}
\text{ and }
{\mathbf b}^{\beta} = \begin{pmatrix} b^{\beta}_1 \\ b^{\beta}_2 \\ b^{\beta}_3 \\ b^{\beta}_4 \end{pmatrix} 
$$

as well as 

$$
{\mathbf c}^{\alpha} = \begin{pmatrix} c^{\alpha}_1 \\ c^{\alpha}_2 \\ c^{\alpha}_3 \\ c^{\alpha}_4 \end{pmatrix}
\text{ and }
{\mathbf c}^{\beta} = \begin{pmatrix} c^{\beta}_1 \\ c^{\beta}_2 \\ c^{\beta}_3 \\ c^{\beta}_4 \end{pmatrix} \, . 
$$

We then have that 

$$
\underline{\underline{A}}_{\alpha\beta}^{(2)} = P_{\alpha\beta} \begin{pmatrix} {\mathbf a}^{\alpha} \\ {\mathbf b}^{\alpha} \\ {\mathbf c}^{\alpha} \end{pmatrix} \begin{pmatrix} {\mathbf a}^{\beta} \\ {\mathbf b}^{\beta} \\ {\mathbf c}^{\beta} \end{pmatrix}^T  \text{ of size } 12 \text{ by } 12 \, . 
$$

This computation is implemented in the notebook in the [notebook](./mom_gmsh_iterate.ipynb).

**Surface-surface interactions contributing to the stiffness matrix** Computing 4D integrals in a double loop over all facets on the boundary of $\Omega$ (double facet iterator in Ferrite.jl). Evaluate 4D integral as double iterated 2D integral. Apply integrals $I_{22}$ for reduction in dimension. 

**Surface-volume and volume-surface interactions contributing to the stiffness matrix** Computing 5D integrals in a nested loop over all tetrahedra and all boundary facets. Evaluate 5D integral as iterated 3D/2D integral. Apply integrals $I_{23}$ for reduction in dimension.

### Section 5.3: Element-by-element Assembly of Stiffness Matrix


**Numerical Computation of Matrix Elements**

Use of adaptive quadrature (cubature) implemented in [hcubature.jl](https://github.com/JuliaMath/HCubature.jl) or alternative) to compute the matrix elements. An example showing the increase of number of evaluations of the integrand is given at [quadgk-examples](https://juliamath.github.io/QuadGK.jl/stable/quadgk-examples/#Integrands-with-singularities-and-discontinuities). An example of Possibly regularization in the kernel;  

**Analytical Method of Matrix Elements**

1. [homogeneous functions](https://en.wikipedia.org/wiki/Homogeneous_function#) 
2. Theorem of [Euler](https://en.wikipedia.org/wiki/Homogeneous_function#Euler's_theorem) for homogeneous functions, reduce dimensionality of integration by one and examples;  
3. approach by Rosen/Cormack  
4. Reduce 6D integrals to 4D integrals
5. reduction of inner integral from 3D to 2D: interchange order of differentiation and integration, obtain two terms, apply Euler on one term;
6. reduction of outer integral from 3D to 2D: derivate of linear shape function is a constant. Apply Green-Gauss divergence theorem; 

$$
\int_a^b f(x) \, dx = F(x) |_{x=a}^{x=b} = F(b) - F(a)
$$

## Section 4: Possible Project Roadmaps  

### Section 1.4: Problem Formulation 

<b>Physical Problem</b> Study problem formulation. Study how the magnetization vector $\mathbf{M}(\mathbf{r})$ change with  of the plate, dimensions of the plate, and the direction of the external magnetic field. Use to this end literature or a reference code.

### Section 2.4: Solution Method  

Explore weak formulation, weighted residual method, Galerkin approximation and linear system formulation for a sequence of problems of increasing complexity.  

1. Poisson equation;
2. one-dimensional Fredholm integro-differential equation. A model problem can be constructed using Sympy;
3. see grad-div equation without kernel;

### Section 3.4: Assembly of the Mass Matrix and Load Vector 

1. loop over elements in the mesh to assemble the mass matrix and the lopad vector. See notebook [mom_hcubature](./mom_hcubature.ipynb); 

### Section 4.4: Numerical Assembly of the Stiffness Matrix 

1. use of quadrature implemented in [hcubature.jl](https://github.com/JuliaMath/HCubature.jl) for integration in 1D (possibly singular, look into number of function evaluatiohs), 2D (reference triangle, coordinate transformation, general triangle), 3D (reference tetrahedra, coordinate transformation, general tetrahedra), 4D (by calling hcubature for 2D twice) and 6D (by calling hcubature for 2D twice). See notebook [mom_hcubature](./mom_hcubature.ipynb);
2. formulate and compute 6D integrals for tetra/tetra interaction assuming no parallel facets. $P_{\alpha}$ with nodes ${\mathbf r}_1 = (0,0,0)$, ${\mathbf r}_2 = (1,0,0)$, ${\mathbf r}_3 = (0,1,0)$ and ${\mathbf r}_4 = (0,0,1)$. 
$P_{\beta}$ with nodes ${\mathbf r}_1 = (0,0,0)$, ${\mathbf r}_2 = (1,0,0)$, ${\mathbf r}_3 = (0.5,1,0)$ and ${\mathbf r}_4 = (0.5,0.5,1)$. Verify that no two faces are parallel to each other;
3. extend previous case to parallel faces. $P_{\alpha}$ reference triangle. $P_{\beta}$ shift of $P_{\alpha}$;
4. apply Euler theorem for homogeneous function and compute resulting 4D integrals numerically; 
5. loop over elements over the mesh to compute the load vector. Assume constant external field. Provide more details (expression of the local vector per element) here. See notebook [mom_gmsh_iterate](./mom_gmsh_iterate.ipynb); 
6. loop over elements over the mesh to compute the mass matrix. Provide more details (expression of the local matrix per element) here;
7. use of hcubature to compute the stiffness matrix elements. Possibly regularization in the kernel;

### Section 5.4: Analytical Assembly of the Stiffness Matrix 

1. implement point-edge interactions (cfr. seperate notebook); 
2. implement edge-edge interactions  (cfr. seperate notebook); 
2. implement point-facet interactions  (cfr. seperate notebook);

## Section 5: The Julia Programming Language

### Introductory Material
- Elementary introduction: [Thinking Julia](https://benlauwens.github.io/ThinkJulia.jl/latest/book.html);
- Course: [Scientific-Programming-in-Julia](https://juliateachingctu.github.io/Scientific-Programming-in-Julia/stable/);  
- Aalto Short Course: [julia-introduction](https://github.com/AaltoRSE/julia-introduction); 
- Video Collection by Chris Rackauckas: [link](https://www.youtube.com/playlist?list=PLCAl7tjCwWyGjdzOOnlbGnVNZk0kB8VSa) 

### Project Specific Packages 

- Numerical Integral using hcubature 
- Mesh Generation using GMSH 
- Variational Formulation and Finite Element Methods: [Minfem.jl](https://minfem.github.io/MinFEM.jl/stable/) 

## References 

### References of the Method of Moments 

(Provide more guidance on what and how to read.) 

1. R. F. Harrington, <i>Field Computation by Moment Methods</i>, Wiley-IEEE Press, 1993. 
2. Morandi, A., Fabbri, M., & Ribani, P. L. (2010). <i>A modified formulation of the volume integral equations method for 3-D magnetostatics</i>. IEEE transactions on magnetics, 46(11), 3848-3859. And similar references;
3. Le-Duc, T., Meunier, G., Chadebec, O., Guichon, J. M., & Bastos, J. P. A. (2013). <i> General integral formulation for the 3D thin shell modeling</i>. IEEE transactions on magnetics, 49(5), 1989-1992. And similar references; 

### References on the weighted residual method, the variational formulation and the finite element method 

1. [Introduction to Numerical Methods for Variational Problems](https://link.springer.com/book/10.1007/978-3-030-23788-2) by Hans Petter Langtangen and Kent-Andre Mardal. The [book](https://link.springer.com/book/10.1007/978-3-030-23788-2) is freely available; 
2. [Wolfgang Bangerth's video lectures](https://www.math.colostate.edu/~bangerth/videos.html); 
3. [wiki Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method): Section 3 for the weak form and Section 4 for the finite element discretization;  
4. [Comsol Multiphysics Finite Element Method](https://www.comsol.com/multiphysics/finite-element-method): more information and illustrations; 
5. [Comsol Multiphysics Brief Introduction to the Weak Form](https://www.comsol.com/blogs/brief-introduction-weak-form): good introduction to a theoretical concept that provides a basis for the finite element method; 
6. [Ferrite Introduction to FEM](https://ferrite-fem.github.io/Ferrite.jl/stable/topics/fe_intro/)

### Reference Computations 

**Reference Tetrahedron** Assume the reference tetrahedron to have the nodes ${\mathbf x}_1 = (0,0,0)$, ${\mathbf x}_2 = (1,0,0)$, ${\mathbf x}_3 = (0,1,0)$ and ${\mathbf x}_4 = (0,0,1)$. Then the shape function are defined as 

$$
\phi_1({\mathbf r}) = 1 - x - y - z \\
\phi_2({\mathbf r}) = x \\
\phi_3({\mathbf r}) = y \\
\phi_4({\mathbf r}) = z 
$$

**Two-Tetrahedra Test Case** For future reference, we define a test case involving two tetrahedra $P_{\alpha}$ and $P_{\beta}$. Assume $P_{\alpha}$ to be the reference element. 
Assume $P_{\beta}$ to be the reference element shifted along the $x$-axis by a distance of $x_0$. Then 

$$
\phi^{\alpha}_1({\mathbf r}) = 1 - x - y - z \\
\phi^{\alpha}_2({\mathbf r}) = x \\
\phi^{\alpha}_3({\mathbf r}) = y \\
\phi^{\alpha}_4({\mathbf r}) = z 
$$

$$
\phi^{\beta}_1({\mathbf r}) = 1 - (x-x_0) - y - z \\
\phi^{\beta}_2({\mathbf r}) = x - x_0 \\
\phi^{\beta}_3({\mathbf r}) = y \\
\phi^{\beta}_4({\mathbf r}) = z 
$$

$$
{\mathbf a}^{\alpha} = \begin{pmatrix} -1 \\ 1 \\ 0 \\ 0 \end{pmatrix}
\hspace{.5cm}
{\mathbf b}^{\alpha} = \begin{pmatrix} -1 \\ 0 \\ 1 \\ 0 \end{pmatrix}
\hspace{.5cm}
{\mathbf c}^{\alpha} = \begin{pmatrix} -1 \\ 0 \\ 0 \\ 1 \end{pmatrix}
$$


### Reference solution for Fredholdm integral equation 

To solve a Fredholm integral equation in SymPy, you cannot use a single "built-in" solver function. Instead, you represent the equation symbolically and use techniques like the Method of Undetermined Coefficients to solve for the unknown function.A classic Fredholm equation of the second kind looks like this:
$$
y(x)=f(x)+\lambda \int _{a}^{b}K(x,t)y(t)\,dt
$$
Here is a complete example solving the Fredholm integral equation:
$$
y(x)=x+\int _{0}^{1}x\cdot t\cdot y(t)\,dt
$$
Step-by-Step Python Code

Since the kernel $K(x, t) = x \cdot t$ is separable, the integral $\int_{0}^{1} t \cdot y(t) \, dt$ evaluates to a constant, let's call it \(C\). This allows us to solve it algebraically.

When you run this script below, SymPy performs the following calculations: It evaluates the definite integral $\int_{0}^{1} t(t + C t) \, dt = \int_{0}^{1} (1+C)t^2 \, dt = \frac{1+C}{3}$ It sets up the linear equation $C = \frac{1+C}{3}$ and solves for $C$, yielding $C = \frac{1}{2}$. It substitutes $C$ back into $y(x) = x + Cx$ to yield the exact analytical solution:$\mathbf{y(x)=}\frac{\mathbf{3x}}{\mathbf{2}}$. 

In [1]:
from sympy import symbols, Integral, solve, Eq

# 1. Define symbols
x, t, C = symbols('x t C')

# 2. Assume the form of y(x) based on the structure of the equation
# Since y(x) = x + x * Integral(t * y(t)), y(x) must be of the form: x + C * x
y_x = x + C * x

# 3. Substitute this assumed form into the integral part to find C
# The constant C is defined as the integral of t * y(t) from 0 to 1
y_t = y_x.subs(x, t)  # Replace x with t for the integration variable
integral_expr = Integral(t * y_t, (t, 0, 1))

# 4. Set up the compatibility equation: C = Integral(t * y(t))
compat_eq = Eq(C, integral_expr.doit())

# 5. Solve for the unknown constant C
C_value = solve(compat_eq, C)[0]

# 6. Substitute C back into our assumed y(x) to get the final solution
final_solution = y_x.subs(C, C_value)

print(f"The constant C is: {C_value}")
print(f"The solution y(x) is: {final_solution}")


LoadError: syntax: extra token "sympy" after end of expression

### Reference Solution for Fredholm integral equation with convolution kernel - Ask singular kernel - Ask code in Ferrite.jl 

To find an analytical solution to a second-order Fredholm integro-differential equation with a convolution kernel, the most reliable algebraic method on a closed interval \([0, 1]\) involves expanding the convolution kernel into a degenerate (separable) form.
Let's walk through the general analytical solution step-by-step using a classic example where the kernel is \(k(x-y) = \sinh(x-y)\) or \(\sin(x-y)\), which can be separated using standard trigonometric or hyperbolic identities.
Step-by-Step Analytical Framework
Consider the constant-coefficient equation on \([0, 1]\):
\(u^{\prime \prime }(x)-u(x)+\int _{0}^{1}\sinh (x-y)u(y)\,dy=f(x)\)
With the boundary conditions: \(u(0) = 0, \quad u(1) = 0\)
Step 1: Separate the Convolution Kernel
Using the hyperbolic identity \(\sinh(x-y) = \sinh(x)\cosh(y) - \cosh(x)\sinh(y)\), we can factor out the variables dependent on \(x\) from the integral:
\(\int _{0}^{1}\sinh (x-y)u(y)\,dy=\sinh (x)\int _{0}^{1}\cosh (y)u(y)\,dy-\cosh (x)\int _{0}^{1}\sinh (y)u(y)\,dy\)
Step 2: Introduce Unknown Constants
Since the integration limits are fixed constants (\(0\) and \(1\)), the definite integrals evaluate to constant values. We define two unknown constants, \(C_{1}\) and \(C_{2}\):
\(C_{1}=\int _{0}^{1}\cosh (y)u(y)\,dy\)
\(C_{2}=\int _{0}^{1}\sinh (y)u(y)\,dy\)
Substituting these back into the original equation converts the integro-differential equation into a standard Ordinary Differential Equation (ODE):
\(u^{\prime \prime }(x)-u(x)+C_{1}\sinh (x)-C_{2}\cosh (x)=f(x)\)
Step 3: Solve the Non-Homogeneous ODE
Rearrange the equation to isolate the differential operator:
\(u^{\prime \prime }(x)-u(x)=f(x)-C_{1}\sinh (x)+C_{2}\cosh (x)\)
The complete solution \(u(x)\) consists of a complementary solution \(u_c(x)\) and a particular solution \(u_p(x)\):
1. Complementary: The characteristic equation \(r^2 - 1 = 0\) gives \(r = \pm 1\), so \(u_c(x) = A \cosh(x) + B \sinh(x)\).
2. Particular: Find \(u_p(x)\) based on the specific form of \(f(x)\) using the method of undetermined coefficients or variation of parameters. Note that since \(\sinh(x)\) and \(\cosh(x)\) are already part of the homogeneous solution, terms multiplying them will require a factor of \(x\) (e.g., \(x \sinh(x)\)).
Apply your boundary conditions (\(u(0)=0, u(1)=0\)) at this stage to express the constants \(A\) and \(B\) in terms of \(C_{1}\), \(C_{2}\), and \(f(x)\). This gives you an expression for \(u(x)\) that depends solely on the numerical values of \(C_{1}\) and \(C_{2}\).
Step 4: Resolve the Constants Matrix
To find the exact values of \(C_{1}\) and \(C_{2}\), substitute your newly found formula for \(u(x)\) back into the definitions of \(C_{1}\) and \(C_{2}\) from Step 2:
\(C_{1}=\int _{0}^{1}\cosh (y)u(y,C_{1},C_{2})\,dy\)
\(C_{2}=\int _{0}^{1}\sinh (y)u(y,C_{1},C_{2})\,dy\)
Evaluating these integrals yields a system of two linear algebraic equations with two variables:
\(\left(\begin{matrix}a_{11}&a_{12}\\ a_{21}&a_{22}\end{matrix}\right)\left(\begin{matrix}C_{1}\\ C_{2}\end{matrix}\right)=\left(\begin{matrix}d_{1}\\ d_{2}\end{matrix}\right)\)
Solving this system gives the precise values for \(C_{1}\) and \(C_{2}\). Substituting them back into your ODE solution yields the exact analytical solution.
Other Analytical Scenarios
• Infinite Domain \((-\infty, \infty)\): If the boundary limits are infinite, the equation is solved using the Fourier Transform, mapping the convolution directly into multiplication: \(-\omega^2 U(\omega) - U(\omega) + \sqrt{2\pi} K(\omega)U(\omega) = F(\omega)\).
• Non-Separable Kernels: If \(k(x-y)\) cannot be separated cleanly (e.g., \(\ln\vert{}x-y\vert{}\)), an exact closed-form analytical solution is rarely possible, and resolvent kernel methods or series expansions (like Taylor or Frobenius series) must be used.
To write out the precise, finalized analytical solution for your problem, could you provide:
• The specific forcing function \(f(x)\) (e.g., \(x^{2}\), \(e^{x}\), \(0\))?
• The exact convolution kernel \(k(x-y)\) you are working with?
• Your boundary/initial values?